# Synthetic Data Generation with OpenAI

This notebook reads `seed_data.csv` and generates additional records using OpenAI. It then filters out duplicates and invalid records before writing the final dataset to `synthetic_data.csv`.

## API key setup with `.env`

Create or edit `.env` in this folder and add your key:

`OPENAI_TOKEN=your_openai_api_key_here`

If you're running in Google Colab, set the Open AI API key using the terminal.

### Validate local secrets file

Run the next code cell to confirm `.env` exists before generation.

In [ ]:
from pathlib import Path

# Check that local secrets file exists in the notebook working directory.
env_path = Path.cwd() / ".env"
if env_path.exists():
    print(f"Found .env at: {env_path}")
else:
    print("Missing .env. Create it with: OPENAI_TOKEN=your_openai_api_key_here")

# Load environment variables from the .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path=env_path)

## Workflow

1. Configure paths and generation settings.
2. Load seed data and define JSON schema.
3. Call OpenAI with Structured Outputs using `text.format` and a JSON schema.
4. Validate locally, remove duplicates, and keep only clean records.
5. Export combined seed + synthetic records to `synthetic_data.csv`.

### Initialize configuration and client

This cell imports required libraries, validates that `OPENAI_TOKEN` exists, and initializes the OpenAI client used later.

In [ ]:
import csv
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Dict, List, Set

from jsonschema import Draft202012Validator
from openai import OpenAI

# Adjust as needed for your use case and desired model
MODEL = "gpt-5.6-luna"
TARGET_NEW_RECORDS = 750
BATCH_SIZE = 10

REPO_ROOT = Path.cwd().resolve().parent
DATASETS_DIR = REPO_ROOT / "datasets"
SEED_PATH = DATASETS_DIR / "seed_data.csv"
OUTPUT_PATH = DATASETS_DIR / "synthetic_data.csv"

api_key = os.getenv("OPENAI_TOKEN")
if not api_key:
    raise EnvironmentError(
        "OPENAI_TOKEN is not set. Add it to .env as OPENAI_TOKEN=your_openai_api_key_here."
    )

client = OpenAI(api_key=api_key)
print(f"Using seed file: {SEED_PATH}")
print(f"Output file: {OUTPUT_PATH}")

### Load seed data and build schemas

This cell loads `seed_data.csv` and builds both record-level and response-level JSON schemas for validation and Structured Outputs.

In [ ]:
# Load the seed table from CSV so the notebook works with flat tabular data.
with SEED_PATH.open("r", encoding="utf-8", newline="") as f:
    seed_records = list(csv.DictReader(f))

if not seed_records:
    raise ValueError("seed_data.csv must contain at least one data row.")

# Define the required output structure explicitly.
required_fields = [
    "customer_id",
    "category",
    "sub_category",
    "channel",
    "tone_urgency",
    "message",
    "prompt",
    "chosen",
    "rejected",
]

record_schema = {
    "type": "object",
    "additionalProperties": False,
    "required": required_fields,
    "properties": {
        "customer_id": {"type": "string", "pattern": "^CUST-[0-9]{4,}$"},
        "category": {"type": "string", "minLength": 1},
        "sub_category": {"type": "string", "minLength": 1},
        "channel": {"type": "string", "minLength": 1},
        "tone_urgency": {"type": "string", "minLength": 1},
        "message": {"type": "string", "minLength": 10},
        "prompt": {"type": "string", "minLength": 10},
        "chosen": {"type": "string", "minLength": 10},
        "rejected": {"type": "string", "minLength": 10},
    },
}

# Structured Outputs requires a top-level object schema.
batch_schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["records"],
    "properties": {
        "records": {
            "type": "array",
            "minItems": 1,
            "maxItems": BATCH_SIZE,
            "items": record_schema,
        }
    },
}

record_validator = Draft202012Validator(record_schema)
print(f"Loaded {len(seed_records)} seed records.")
print("Schema structure established for required fields.")

### Define validation and deduplication helpers

This cell defines helper functions for text normalization, record fingerprinting, strict validation, and extraction of the `records` payload from model output.

In [ ]:
# Normalizes text before comparisons so duplicate checks are case/whitespace insensitive.
def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.strip().lower())


# Creates a canonical fingerprint from key semantic fields used for duplicate validation.
def record_fingerprint(record: Dict[str, Any]) -> str:
    significant_fields = [
        "category",
        "sub_category",
        "channel",
        "tone_urgency",
        "message",
        "prompt",
        "chosen",
        "rejected",
    ]
    # Canonical fingerprint reduces false negatives from minor spacing/case changes.
    canonical = "|".join(normalize_text(str(record.get(k, ""))) for k in significant_fields)
    return canonical


# Validates each record against JSON schema and ensures chosen/rejected are not identical.
def validate_record(record: Dict[str, Any]) -> List[str]:
    errors = [err.message for err in record_validator.iter_errors(record)]
    if record.get("chosen", "").strip() == record.get("rejected", "").strip():
        errors.append("chosen and rejected responses must differ")
    return errors


# Validates payload shape: top-level JSON object with a records array.
def parse_candidate_payload(output_text: str) -> List[Dict[str, Any]]:
    data = json.loads(output_text)
    if not isinstance(data, dict):
        raise ValueError("Model output must be a JSON object.")

    records = data.get("records")
    if not isinstance(records, list):
        raise ValueError("Model output object must contain a 'records' array.")

    return records


# Enforces customer_id format by scanning valid IDs and incrementing the max numeric suffix.
def next_customer_id(existing_ids: Set[str]) -> str:
    max_num = 1000
    for item in existing_ids:
        match = re.match(r"^CUST-(\d+)$", item)
        if match:
            max_num = max(max_num, int(match.group(1)))
    return f"CUST-{max_num + 1:04d}"


existing_ids = {str(r.get("customer_id", "")) for r in seed_records}
known_fingerprints = {record_fingerprint(r) for r in seed_records}
print(f"Indexed {len(known_fingerprints)} unique seed fingerprints.")

### Generate synthetic records with Structured Outputs

This cell calls the Responses API in batches, enforces schema-shaped output, rejects invalid or duplicate rows, and assigns new customer IDs to accepted records.

In [ ]:
SYSTEM_PROMPT = """You generate high-quality synthetic customer service preference records.
Return realistic records that match the seed format and style, but do not copy seed examples.
Always return valid JSON that matches the provided schema.
"""


def build_user_prompt(seed_examples: List[Dict[str, Any]], batch_size: int) -> str:
    few_shot = random.sample(seed_examples, k=min(3, len(seed_examples)))
    return (
        f"Generate {batch_size} new records inside the JSON object field records. "
        "Return only JSON and no markdown. "
        "Vary situations and wording while preserving domain realism.\n\n"
        f"Seed examples:\n{json.dumps(few_shot, indent=2)}"
    )


def extract_output_text(response: Any) -> str:
    output_text = getattr(response, "output_text", None)
    if output_text:
        return output_text

    output = getattr(response, "output", [])
    parts: List[str] = []
    for item in output:
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                parts.append(content.get("text", ""))
    joined = "".join(parts).strip()
    if not joined:
        raise ValueError("Could not extract text output from OpenAI response.")
    return joined


def request_batch(batch_size: int) -> List[Dict[str, Any]]:
    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(seed_records, batch_size)},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "synthetic_customer_records",
                "schema": batch_schema,
                "strict": True,
            }
        },
    )
    raw = extract_output_text(response)
    return parse_candidate_payload(raw)


generated_records: List[Dict[str, Any]] = []
rejected_duplicates = 0
rejected_invalid = 0

attempt = 0
while len(generated_records) < TARGET_NEW_RECORDS:
    attempt += 1
    remaining = TARGET_NEW_RECORDS - len(generated_records)
    current_batch = min(BATCH_SIZE, remaining)

    try:
        candidates = request_batch(current_batch)
    except Exception as exc:
        print(f"Attempt {attempt}: request failed: {exc}")
        continue

    accepted_this_round = 0
    for record in candidates:
        errors = validate_record(record)
        if errors:
            rejected_invalid += 1
            continue

        fingerprint = record_fingerprint(record)
        if fingerprint in known_fingerprints:
            rejected_duplicates += 1
            continue

        # Assign a fresh sequential ID for accepted synthetic records.
        record["customer_id"] = next_customer_id(existing_ids)
        existing_ids.add(record["customer_id"])
        known_fingerprints.add(fingerprint)
        generated_records.append(record)
        accepted_this_round += 1

        if len(generated_records) >= TARGET_NEW_RECORDS:
            break

    print(
        f"Attempt {attempt}: accepted={accepted_this_round}, "
        f"total_new={len(generated_records)}/{TARGET_NEW_RECORDS}"
    )

print("Generation complete.")
print(f"Accepted: {len(generated_records)}")
print(f"Rejected duplicates: {rejected_duplicates}")
print(f"Rejected invalid: {rejected_invalid}")

### Save final dataset

This cell combines seed and accepted synthetic records, writes the final CSV output file, and prints a sample generated record for a quick quality check.

In [ ]:
# Combine curated seed data with newly accepted synthetic records.
final_dataset = [*seed_records, *generated_records]

with OUTPUT_PATH.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=required_fields)
    writer.writeheader()
    writer.writerows(final_dataset)

print(f"Wrote {len(final_dataset)} total records to {OUTPUT_PATH}")
print(f"New records added: {len(generated_records)}")

if generated_records:
    print("Sample generated record:")
    print(json.dumps(generated_records[0], indent=2, ensure_ascii=False))

In [15]:
print(final_dataset[300]["message"])

print(final_dataset[300]["chosen"])

print(final_dataset[300]["rejected"])

The steak in my burrito was tough and cold when I picked up my order from the Riverside location. I expected better for the price.
I’m sorry your steak burrito was tough and cold when you picked it up at our Riverside location. That falls short of our quality standards, and we’d appreciate your order details and contact information at [Manager Contact] so we can review the preparation and make this right.
That is probably how our steak is supposed to taste. If you wanted a hot meal, you should have eaten it immediately instead of complaining about the price.
